# EDA, preprocessing dan BERTopic

Jalankan seluruh sel berurutan dari root proyek. Notebook ini membaca `data/raw/raw.csv`, membersihkan dan membagi teks, lalu menjalankan BERTopic pada kalimat unik. Setiap kemunculan per artikel tetap disimpan dan dihubungkan kembali ke hasil topik lewat `id_teks`. Tidak ada stemming, lowercasing, atau stopword removal pada teks untuk embedding.

**Lingkungan:** pilih kernel Python yang memiliki `pandas`, `Sastrawi`, `sentence-transformers`, `umap-learn`, `hdbscan` dan `bertopic`. Bila belum tersedia, instal ke lingkungan kernel tersebut, misalnya `%pip install pandas Sastrawi sentence-transformers umap-learn hdbscan bertopic`. Model embedding akan diunduh pada pemakaian pertama. Pelatihan bisa membutuhkan waktu dan RAM besar.

Keluaran versi baru disimpan di `data/processed/bertopic_prep/`; file dan topik lama tidak ditimpa. File di folder keluaran ini diperbarui ketika notebook dijalankan ulang.

In [1]:
import re
import unicodedata
from pathlib import Path
import pandas as pd

SOURCE = Path("data/raw/raw.csv")
OUT = Path("data/processed/bertopic_prep")
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.read_csv(SOURCE, dtype={"id_artikel": str}, keep_default_na=False)
required = {"id_artikel", "judul", "isi_mentah", "timestamp"}
if not required.issubset(raw.columns) or raw["id_artikel"].duplicated().any() or raw["id_artikel"].eq("").any():
    raise ValueError("Kolom wajib tidak lengkap, atau id_artikel kosong/duplikat.")
print(f"Artikel: {len(raw):,}; isi kosong: {raw['isi_mentah'].str.strip().eq('').sum():,}")

Artikel: 3,979; isi kosong: 0


## Pecah kalimat dan bersihkan

Pemecah kalimat berbasis aturan melindungi singkatan, inisial, dan angka desimal/ribuan. Pemotongan kalimat tidak sempurna; cek laporan flag. `id_kalimat` memakai posisi hasil pemecahan versi ini, sehingga **tidak boleh dianggap sama** dengan ID yang sama di keluaran notebook lama.

In [2]:
SINGKATAN = ["PT", "Tbk", "Persero", "Rp", "US", "No", "Jl", "Dr", "Ir", "Hj", "Prof", "Sdr", "S.E", "M.M", "Ph.D", "dll", "dsb", "dkk", "a.l", "yakni"]
SPLIT = re.compile(r'(?<=[.!?])\s+|(?<=[.!?]["”])\s+')
BANK = {
    "BCA": r"\bbca\b|\bbbca\b|bank central asia",
    "BRI": r"\bbri\b|\bbbri\b|bank rakyat indonesia|\bbrilink\b",
    "BNI": r"\bbni\b|\bbbni\b|bank negara indonesia",
    "Mandiri": r"\bmandiri\b|\bbmri\b|bank mandiri",
}
DATELINE = re.compile(r'^\s*(?:jakarta\s*,?\s*)?cnbc\s+indonesia\s*[-–—:]\s*', re.I)
URL = re.compile(r'(?i)\b(?:https?://|www\.)[^\s<>"”]+')
# Hanya pola promosi atau footer yang jelas; teks liputan yang menyebut CNBC dipertahankan.
FOOTER = re.compile(r'^(?:cnbc\s+indonesia\s+research(?:\s+market@cnbcindonesia\.com)?|selamat\s+berinvestasi\s+secara\s+bijak)[.!\s]*$', re.I)
LINK_PROMO = re.compile(r'(?i)(?:baca\s+artikel\s+cnbc\s+indonesia|selengkapnya\s+di\s+sini).*(?:cnbcindonesia\.com|download\s+apps\s+cnbc)')
DISCLAIMER = re.compile(r"^keputusan berinvestasi sepenuhnya berada di tangan masing-masing investor sesuai dengan profil risiko dan tujuan keuangan pribadi[.!\s]*$", re.I)
FRAGMENT_ONLY = re.compile(r"^(?:ltd\.?|\(?dok\.?)$", re.I)
NUMERIC_ONLY = re.compile(r'^[\W\d_]+$', re.UNICODE)
INVISIBLE = re.compile('[\u200b\u200c\u200d\u200e\u200f\u2060\ufeff]')

def split_sentences(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    for ab in SINGKATAN:
        text = re.sub(rf"\b{re.escape(ab)}\.", lambda m: m.group(0)[:-1] + "<DOT>", text)
    text = re.sub(r"\b([A-Z])\.(?=\s)", r"\1<DOT>", text)
    text = re.sub(r"(?<=\d)\.(?=\d)", "<DOT>", text)
    return [x.replace("<DOT>", ".").strip() for x in SPLIT.split(text) if x.strip()]

def clean_sentence(text):
    # Deteksi promo sebelum URL dihapus.
    text = unicodedata.normalize("NFC", str(text))
    text = INVISIBLE.sub("", text)
    text = re.sub(r"\s+", " ", text).strip()
    if LINK_PROMO.search(text):
        return "", "promosi_tautan"
    text = DATELINE.sub("", text)
    text = re.sub(r'^\s*[–—•·]+\s*', '', text)
    text = URL.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return "", "kosong_setelah_bersih"
    if FOOTER.fullmatch(text):
        return "", "footer"
    if DISCLAIMER.fullmatch(text):
        return "", "disclaimer_investasi"
    if FRAGMENT_ONLY.fullmatch(text):
        return "", "fragmen_nonkalimat"
    if NUMERIC_ONLY.fullmatch(text):
        return "", "nomor_atau_tanda_saja"
    return text, ""

rows = []
for row in raw.itertuples(index=False):
    combined = f"{row.judul} {row.isi_mentah}"
    banks = ", ".join(bank for bank, pattern in BANK.items() if re.search(pattern, combined, flags=re.I))
    for index, original in enumerate(split_sentences(row.isi_mentah)):
        cleaned, reason = clean_sentence(original)
        rows.append({"id_artikel": row.id_artikel, "id_kalimat": f"{row.id_artikel}_{index}",
                     "bank": banks, "timestamp": row.timestamp, "kalimat_asli": original,
                     "kalimat": cleaned, "alasan_dibuang": reason})
all_rows = pd.DataFrame(rows)
removed = all_rows[all_rows.alasan_dibuang.ne("")].copy()
occ = all_rows[all_rows.alasan_dibuang.eq("")].copy()
occ["n_kata"] = occ.kalimat.str.split().str.len()
# Flag adalah petunjuk review, bukan ground truth dan bukan kriteria penghapusan otomatis.
occ["flag_pendek"] = occ.n_kata.lt(4)
occ["flag_panjang"] = occ.n_kata.gt(80)
occ["flag_konteks_daftar"] = occ.kalimat.str.contains(r"(?i)\b(?:tbk|persero)\b.*(?:rp|us\$).*\b\d{1,2}[.)]\s*$", regex=True, na=False)
occ["flag_sisa_markup"] = occ.kalimat.str.contains(r"(?i)\[gambas:|\b(?:foto|infografis)\s*:|\bbaca\s+(?:juga|artikel)\b", regex=True, na=False)
# Group berdasarkan teks utuh yang sudah dinormalisasi spasi/NFC, tanpa menyamakan huruf besar atau angka.
occ["id_teks"] = occ.groupby("kalimat", sort=False)["id_kalimat"].transform("first")
unique = occ.drop_duplicates("id_teks").copy()
counts = occ.groupby("id_teks").size().rename("jumlah_kemunculan")
unique = unique.merge(counts, left_on="id_teks", right_index=True, validate="one_to_one")
assert occ.id_kalimat.is_unique and unique.id_teks.is_unique
assert len(occ) == int(unique.jumlah_kemunculan.sum())
print(f"Potongan awal: {len(all_rows):,} | dibuang: {len(removed):,} | kemunculan tersimpan: {len(occ):,} | teks unik: {len(unique):,}")
print(removed.alasan_dibuang.value_counts().to_string())
print("Flag (bisa tumpang tindih):")
print(occ[["flag_pendek", "flag_panjang", "flag_konteks_daftar", "flag_sisa_markup"]].sum().to_string())

Potongan awal: 82,509 | dibuang: 679 | kemunculan tersimpan: 81,830 | teks unik: 65,246
alasan_dibuang
nomor_atau_tanda_saja    425
footer                   170
disclaimer_investasi      47
fragmen_nonkalimat        30
promosi_tautan             7
Flag (bisa tumpang tindih):
flag_pendek             345
flag_panjang            842
flag_konteks_daftar    1624
flag_sisa_markup         30


## Simpan dan tinjau

`perubahan_preprocessing.csv` mencatat teks asli dan hasil pembersihan; `dibuang_preprocessing.csv` mencatat potongan yang dibuang beserta alasannya. File review besar karena berisi teks penuh dan tetap berada di lokal; proposal melarang distribusi ulang teks artikel penuh. Periksa sampel flag dan hasil pembersihan sebelum BERTopic dijalankan ulang.

In [3]:
occ.to_csv(OUT / "sentences_bertopic_occurrences.csv", index=False, encoding="utf-8-sig")
unique.to_csv(OUT / "sentences_bertopic_unique.csv", index=False, encoding="utf-8-sig")
removed.to_csv(OUT / "dibuang_preprocessing.csv", index=False, encoding="utf-8-sig")
changed = occ.loc[occ.kalimat.ne(occ.kalimat_asli), ["id_artikel", "id_kalimat", "kalimat_asli", "kalimat"]]
changed.to_csv(OUT / "perubahan_preprocessing.csv", index=False, encoding="utf-8-sig")
flags = occ.loc[occ[["flag_pendek", "flag_panjang", "flag_konteks_daftar", "flag_sisa_markup"]].any(axis=1)]
flags.to_csv(OUT / "perlu_review.csv", index=False, encoding="utf-8-sig")
print(f"Perubahan teks: {len(changed):,}; perlu review: {len(flags):,}")
print("Lokasi:", OUT.resolve())

Perubahan teks: 1,345; perlu review: 2,812
Lokasi: D:\Kuliah\Semester 5\Text Mining\AOL Project\Bank-News-Topic-Modelling-Aspect-Based-Sentimental-Analysis\data\processed\bertopic_prep


## EDA singkat dan review kualitas

Flag bukan vonis noise: kalimat pendek, panjang, dan potongan daftar tetap dipertahankan untuk review. `perlu_review.csv`, `dibuang_preprocessing.csv`, dan `perubahan_preprocessing.csv` menyimpan bukti. Label bank berdasarkan artikel, bukan bukti bahwa setiap kalimat membahas bank tersebut. ID kalimat versi ini tidak boleh dicampurkan dengan ID notebook lama meski bentuknya sama.

In [4]:
print("Rentang tanggal artikel:", raw.timestamp.min(), "sampai", raw.timestamp.max())
print("\nStatistik panjang kalimat unik:")
print(unique.n_kata.describe().round(1).to_string())
print("\nTeks identik dengan kemunculan terbanyak (tidak otomatis noise):")
for row in unique.sort_values("jumlah_kemunculan", ascending=False).head(10).itertuples():
    print(f"{row.jumlah_kemunculan:>3}x | {row.kalimat}")
print("\nContoh yang ditandai untuk review:")
for text in flags.kalimat.head(10):
    print(" -", text)

Rentang tanggal artikel: 2024-10-21 sampai 2026-10-20

Statistik panjang kalimat unik:
count    65246.0
mean        19.9
std         14.9
min          1.0
25%         12.0
50%         17.0
75%         23.0
max        448.0

Teks identik dengan kemunculan terbanyak (tidak otomatis noise):
 27x | Nasabah BRI Prioritas bisa mendapatkan BRI Prioritas Debit Card yang menyediakan semua layanan perbankan, seperti perencanaan keuangan dan investasi hingga perencanaan dana pensiun.
 23x | Mengutip Stockbit, berikut net foreign sell perdagangan Rabu!
 18x | "Dalam sekejap, seluruh mobil di garasi terbakar, termasuk juga seisi rumah.
 18x | Yakni welcome bonus berupa cashback senilai Rp 1,5 juta setelah aktivasi dan transaksi pertama.
 18x | "Perusahaan para cukong dan keluarga Soeharto merupakan sasaran utama pembakaran dan penjarahan.
 18x | Keuntungan bagi nasabah prioritas Bank Mandiri dapat dirasakan sejak bergabung.
 18x | Nasabah istimewa BNI Emerald akan mendapatkan produk dan layanan yan

## Stopword untuk kata kunci topik

Stopword hanya diterapkan pada CountVectorizer/c-TF-IDF; `kalimat` tetap utuh untuk embedding. Negasi tetap dipertahankan. Daftar dan parameter di sini merupakan baseline yang bisa dibandingkan dengan percobaan tuning berikutnya.

In [5]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stop_generik = set(StopWordRemoverFactory().get_stop_words())
stop_domain = {"rp", "pt", "tbk", "persero", "miliar", "triliun", "juta", "ribu",
               "i", "ii", "iii", "yoy", "as", "us", "senin", "selasa", "rabu",
               "kamis", "jumat", "sabtu", "minggu"}
negasi = {"tidak", "tak", "bukan", "belum", "jangan", "tanpa", "kurang", "non",
          "anti", "minus", "gagal", "batal", "turun", "melemah", "koreksi"}
stopwords_id = sorted((stop_generik | stop_domain) - negasi)
assert not (set(stopwords_id) & negasi)
print(f"Stopword untuk representasi topik: {len(stopwords_id)}; negasi dipertahankan.")

Stopword untuk representasi topik: 141; negasi dipertahankan.


## BERTopic

Baseline memakai parameter lama: `min_cluster_size=150`, `min_samples=10`, `min_df=5`, seed 42. Ini **bukan hasil tuning**. Untuk membandingkan percobaan secara adil, gunakan korpus yang sama dan simpan hasil setiap percobaan sebelum menjalankan ulang notebook. Nomor topik baru tidak sama dengan nomor topik lama.

In [6]:
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

SEED = 42
MIN_CLUSTER_SIZE = 150
MIN_SAMPLES = 10
MIN_DF = 5
MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
sent = unique.copy().reset_index(drop=True)
docs = sent["kalimat"].tolist()
if not docs or any(not isinstance(doc, str) or not doc.strip() for doc in docs):
    raise ValueError("Input BERTopic kosong atau mengandung kalimat kosong.")
print(f"Input BERTopic: {len(docs):,} kalimat unik")

c:\Users\frederick.clarence\anaconda3\envs\text-mining-aol\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input BERTopic: 65,246 kalimat unik


In [7]:
embedding_model = SentenceTransformer(MODEL_NAME)
embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                  metric="cosine", random_state=SEED)
hdbscan_model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=MIN_SAMPLES,
                        metric="euclidean", cluster_selection_method="eom", prediction_data=True)
vectorizer_model = CountVectorizer(stop_words=stopwords_id, ngram_range=(1, 2), min_df=MIN_DF)
topic_model = BERTopic(embedding_model=embedding_model, umap_model=umap_model,
                       hdbscan_model=hdbscan_model, vectorizer_model=vectorizer_model,
                       language="multilingual", calculate_probabilities=False, verbose=True)
topics, probs = topic_model.fit_transform(docs, embeddings)
if len(topics) != len(sent):
    raise ValueError("Jumlah hasil topik tidak sama dengan jumlah kalimat input.")
info = topic_model.get_topic_info()
n_out = int((pd.Series(topics) == -1).sum())
print(f"Topik non-outlier: {sum(info.Topic != -1)} | outlier: {n_out:,} ({n_out / len(topics):.1%})")
display(info.head(30))

Batches:   4%|▎         | 38/1020 [02:03<53:13,  3.25s/it]  


KeyboardInterrupt: 

## Preview dan simpan hasil

Preview tidak memotong kalimat sehingga angka yang berbeda tetap terlihat. Dokumen representatif bukan pengganti contoh acak; validasi pemetaan aspek tetap perlu dilakukan.

In [ ]:
for t in info["Topic"].tolist():
    if t == -1:
        continue
    keyword = ", ".join(word for word, _ in (topic_model.get_topic(t) or [])[:8])
    print(f"\n=== Topik {t} (n={int(info.loc[info.Topic == t, 'Count'].iloc[0])}) ===")
    print("Kata kunci:", keyword)
    for text in (topic_model.get_representative_docs(t) or [])[:3]:
        print(" -", text)

In [ ]:
sent["topic"] = topics
if sent.id_teks.duplicated().any():
    raise ValueError("ID teks unik ternyata duplikat.")
result = occ.merge(sent[["id_teks", "topic"]], on="id_teks", how="left", validate="many_to_one")
if result.topic.isna().any() or len(result) != len(occ):
    raise ValueError("Ada kemunculan tanpa topik atau jumlah baris berubah.")
sent.to_csv(OUT / "sentences_unique_with_topic.csv", index=False, encoding="utf-8-sig")
result.to_csv(OUT / "sentences_occurrences_with_topic.csv", index=False, encoding="utf-8-sig")
info.to_csv(OUT / "topic_info_baru.csv", index=False, encoding="utf-8-sig")
print("Hasil topik unik:", OUT / "sentences_unique_with_topic.csv")
print("Hasil topik per kemunculan/artikel:", OUT / "sentences_occurrences_with_topic.csv")
print("Ringkasan topik:", OUT / "topic_info_baru.csv")
print(f"Kemunculan terpetakan: {len(result):,} | artikel: {result.id_artikel.nunique():,}")

## Langkah berikutnya

Tinjau kata kunci dan contoh acak dari setiap topik, validasi relevansi dengan konteks artikel, lalu bandingkan baseline dengan kandidat tuning. Simpan hasil tiap percobaan agar tidak tertimpa. Baru setelah itu gabungkan klaster semakna dan sepakati daftar aspek. Jangan menganggap topik atau `Lainnya` otomatis benar untuk seluruh anggotanya.